In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from ariel_pred.dataset import DataLoaderAndCalibrator
from ariel_pred.preprocessing import SGSmoothing
from ariel_pred.plots import plot_white_curve
from ariel_pred.transit import FunctionFittingBasedPhaseDetector
from ariel_pred.models import TransitMultiplicationFactorFinder, SValuesCNN, SValuesCNNTrainer
from ariel_pred.features import WavelengthsGroupsMultiplierFinder
from ariel_pred.metrics import score, gll
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import random
from scipy.optimize import minimize
from tqdm.auto import tqdm
import pandas as pd
import torch
from torch import nn

In [3]:
data_loader = DataLoaderAndCalibrator(
    data_path=Path("../data/raw"),
    output_path=Path("../data/calibrated/full"),
    force_recalibration=False,
    cut_airs_channels=True,
    binning=4,
    n_jobs=4
)
train_data, train_labels = data_loader.load_all_train_data()
train_data.shape, train_labels.shape

Loading calibrated train data...


((1100, 1406, 283), (1100, 283))

In [4]:
features_extractor = WavelengthsGroupsMultiplierFinder()

features = features_extractor.extract_features(
    train_data,
    average_cross_groups=False,
    wavelengths_groups=[1, 2, 4, 8, 16, 32, 64],
    weights=[1, 1, 1, 1, 1, 1, 1]
)

features.shape

100%|██████████| 1100/1100 [03:29<00:00,  5.26it/s]


(1100, 283, 7)

In [5]:
cnn_train_data = features.transpose(0, 2,1)
cnn_train_data.shape

(1100, 7, 283)

In [52]:
trainer = SValuesCNNTrainer(
    Path("../models/s_values_cnn"),
    device=torch.device("mps"),
    in_channels=cnn_train_data.shape[1],
    num_channels=283
)

In [145]:
trainer = SValuesCNNTrainer(
    Path("../models/s_values_cnn"),
    device=torch.device("mps"),
    in_channels=cnn_train_data.shape[1],
    num_channels=283,
    train_multiplier=1e3
)

res = trainer.train(cnn_train_data, train_labels, n_splits=5, return_predictions=True)

if res is not None:
    spectrum, sigma, (val_rmse, val_gll, val_loss, train_loss) = res # type: ignore

No saved models found in the specified path. Starting training from scratch.


Training Model:   0%|          | 0/1000 [00:00<?, ?it/s]

In [146]:
np.mean((spectrum - train_labels)**2)**0.5

np.float64(0.0007237995244922738)

In [147]:
np.mean((features.mean(axis=2) - train_labels)**2)**0.5

np.float64(0.001371322598468874)

In [148]:
gll(np.concatenate((spectrum, sigma), axis=1), train_labels)

0.3424008066843431

In [149]:
gll(np.concatenate((spectrum, np.ones_like(spectrum) * 0.0007), axis=1), train_labels)

0.33902903984585336

In [150]:
def cost_function(params: np.ndarray) -> float:
    sigma = params[0]
    return -gll(np.concatenate((spectrum, np.ones_like(spectrum) * sigma), axis=1), train_labels)

initial_guess = np.array([0.0004])
result = minimize(cost_function, initial_guess, method='Nelder-Mead')
result.x, -result.fun

(array([0.0007]), np.float64(0.3390290398458534))